# DocuMind - OCR: teach the reader of the text itself (TrOCR) with your three datasets

The detectors (YOLO) say **where** a value is; the OCR says **what** is written there. Tesseract is general-purpose: it drops accents, mixes
`0` / `9` / `8`, loses handwriting and capitals. Here a recognition model that already reads text (**TrOCR**) is fine-tuned on the text-line crops
cut out of your datasets: **receipts** and **cheques** (made by your generators) and **FATURA** invoices.

```
STEP 1  prepare_ocr   cut every labelled box out of the pictures, with its text   ->  data/processed/ocr_crops   (+ Tesseract's score = the number to beat)
STEP 2  train_ocr     fine-tune TrOCR, keep the best epoch                          ->  models/ocr_trocr
STEP 3  evaluate_ocr  Tesseract vs the new model on crops neither was tuned on     ->  reports/ocr_report.json
STEP 4  use it        nothing to change: the receipt and cheque readers pick models/ocr_trocr up by themselves
```
Run it top to bottom with the kernel of your GPU environment. A GPU is needed for step 2 (hours on a CPU).

In [ ]:
import json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image
# Make the project root importable when the notebook is opened from a subfolder.
ROOT = Path.cwd()
while not (ROOT / "documind").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if (ROOT / "documind").is_dir():
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print(f"project root: {ROOT}")
else:
    raise FileNotFoundError(
        "Could not find the project root containing 'documind/'. "
        "Open the notebook from the repo root or install the project with: %pip install -e ."
    )

from documind.config import MODELS_DIR, PROCESSED_DIR, RAW_DIR, REPORTS_DIR

If a package is missing (first run): `pip install -r requirements-ocr.txt` in a terminal with the GPU environment activated (transformers, sentencepiece, accelerate). The first training downloads the pretrained model (about 250 MB) from Hugging Face: internet needed once.

## GPU check (run this first)
Ultralytics only uses the GPU if PyTorch is a **CUDA build** and the NVIDIA driver works. `pip install ultralytics` often installs the CPU-only PyTorch on Windows, so check before training.

If the check below stops with an error, fix it in a terminal with `ai_env` activated, then **restart the kernel**:
```
pip uninstall -y torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

In [ ]:
import torch

REQUIRE_GPU = True   # True = stop here if no GPU is found (instead of silently training for hours on the CPU)

print("python executable :", sys.executable)          # must be inside your ai_env
print("torch             :", torch.__version__)       # ending with +cpu = CPU-only build
print("torch CUDA build  :", torch.version.cuda)      # None = CPU-only build
print("cuda available    :", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu               : {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")
    DEVICE = 0
else:
    DEVICE = "cpu"
    msg = ("No usable GPU for PyTorch. Check, in this order:\n"
           "  1. torch version ends with '+cpu' / CUDA build is None -> reinstall the CUDA build (cell above)\n"
           "  2. python executable is not your ai_env -> Kernel > Change kernel\n"
           "  3. 'nvidia-smi' fails in a terminal -> update the NVIDIA driver")
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print("WARNING (training will run on the CPU):\n" + msg)

## Settings

In [ ]:
from documind.data import prepare_ocr
from documind.training import evaluate_ocr, train_ocr

BASE = "microsoft/trocr-small-printed"   # fast. For the handwriting of the cheques try "microsoft/trocr-base-handwritten" (BATCH = 8)
SOURCES = None                           # None = everything that was prepared; or ["cheques"], ["receipts", "fatura"] ...
EPOCHS = 4
BATCH = 16                               # lower on 'CUDA out of memory' (8 for the base models)
LR = 4e-5
PATIENCE = 2                             # stop after this many epochs without improvement (the best epoch is the one kept)
MAX_PER_SOURCE = 60000                   # training crops kept per source (balanced between the field types)
TORCH_DEVICE = "cpu" if DEVICE == "cpu" else "cuda"
CROPS = prepare_ocr.DEFAULT_OUT

## STEP 1 - the crops and the number to beat
Reads `data/raw/receipts` (or `receipt_dataset`), `data/raw/cheques` (or `cheque_dataset`) and `data/raw/FATURA`; a folder that does not exist is skipped (the message says which).
`check=100` also reads 100 validation crops per source with Tesseract: that is the **score the trained model has to beat**. Takes a few minutes. **Look at the preview picture:** every crop must sit on its text, and the text next to it must be what is written.

In [ ]:
stats = prepare_ocr.run(CROPS, max_per_source=MAX_PER_SOURCE, check=100)
display(pd.Series(stats["counts"], name="crops").to_frame())
print("characters:", stats["n_chars"], "| non-ASCII:", stats["non_ascii"] or "none", "| longest text:", stats["max_len"])
for s, b in stats.get("tesseract_baseline", {}).items():
    print(f"Tesseract on {s}: CER {b['all']['cer']}  exact {b['all']['exact']}")
plt.figure(figsize=(9, 11)); plt.imshow(Image.open(REPORTS_DIR / "ocr_crops_preview.png")); plt.axis("off"); plt.show()

If FATURA crops look shifted in the preview, the box convention was guessed wrong: run `prepare_ocr` from a terminal with `--fatura-mode xywh` (or `xyxy`).

## STEP 2 - a one-minute check, then the real training
The smoke run trains on 64 crops for one step sequence, scores and saves: if it prints `SMOKE OK`, the long run will not die on a typo.

In [ ]:
train_ocr.train(CROPS, base=BASE, sources=SOURCES, smoke=True, device=TORCH_DEVICE, log_path=REPORTS_DIR / "ocr_smoke_log.json", out=MODELS_DIR / "ocr_smoke")

In [ ]:
result = train_ocr.train(CROPS, base=BASE, sources=SOURCES, epochs=EPOCHS, batch=BATCH, lr=LR, patience=PATIENCE, device=TORCH_DEVICE)

In [ ]:
hist = result["history"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot([h["epoch"] for h in hist], [h["train_loss"] for h in hist], marker="o"); ax[0].set_title("training loss")
for s in hist[0]["val"]["by_source"]:
    ax[1].plot([h["epoch"] for h in hist], [h["val"]["by_source"][s]["cer"] for h in hist], marker="o", label=s)
ax[1].set_title("validation CER (lower is better)"); ax[1].legend(); plt.tight_layout(); plt.show()

## STEP 3 - Tesseract against the trained model
On `test` crops (never used for training or for choosing the best epoch). **CER** = share of characters wrong, **exact** = share of crops read without a single mistake.

Careful: the crops come from the same generators as the training crops, so a great score says the model learned these documents, not that it reads every real one. Try real pictures at the end.

In [ ]:
rows = evaluate_ocr.sample_rows(CROPS, "test", 800)
hyps = {"tesseract": evaluate_ocr.read_all("tesseract", rows, None, 1), "trocr": evaluate_ocr.read_all("trocr", rows, MODELS_DIR / "ocr_trocr", 4)}
res = evaluate_ocr.table(rows, hyps)
lines = []
for s, d in res["tesseract"]["sources"].items():
    for k, v in {"ALL": d["all"], **d["kinds"]}.items():
        t = res["trocr"]["sources"][s]["all" if k == "ALL" else "kinds"]
        t = t if k == "ALL" else t[k]
        lines.append({"source": s, "field": k, "n": v["n"], "CER tesseract": v["cer"], "CER trained": t["cer"], "exact tesseract": v["exact"], "exact trained": t["exact"]})
display(pd.DataFrame(lines))
print("ALL sources  CER tesseract", res["tesseract"]["all"]["cer"], "-> trained", res["trocr"]["all"]["cer"])

In [ ]:
from documind.ocr.metrics import cer
worst = sorted(zip(rows, hyps["trocr"], hyps["tesseract"]), key=lambda t: -cer(t[0]["text"], t[1]))[:6]
fig, ax = plt.subplots(len(worst), 1, figsize=(9, 1.3 * len(worst)))
for a, (r, h, t) in zip(ax, worst):
    a.imshow(Image.open(r["path"])); a.axis("off")
    a.set_title(f"truth: {r['text'][:45]}  |  trained: {h[:45]}  |  tesseract: {t[:45]}", fontsize=8, loc="left")
plt.tight_layout(); plt.show()

## STEP 4 - use it: both engines together
Nothing to change in the code. When `models/ocr_trocr` exists, the receipt, cheque and contract readers (`ocr_box`) read every box with **both** the trained model and Tesseract and keep the better reading (`documind/ocr/ensemble.py`): the same text wins; then a cross-check (a cheque's amount in words must be its amount in figures); then the format of the field (a date must be a date, an amount must look like an amount, an IBAN must have its country's length ...); when both look fine and differ, the trained model decides. Control it with an environment variable:

| `DOCUMIND_OCR` | meaning |
|---|---|
| `auto` (default) | both engines together if `models/ocr_trocr` exists and `torch` + `transformers` are installed, else Tesseract alone |
| `both` | the same, but an error if the model is missing |
| `trocr` | the trained model only |
| `tesseract` | Tesseract only (to compare) |

`DOCUMIND_OCR_PRIMARY=tesseract` lets Tesseract decide instead when both readings look fine. For the **invoices**, Tesseract still finds the words; `DOCUMIND_OCR_REFINE=lowconf` lets the trained model re-read the doubtful ones and the same arbitration keeps the better reading (off by default - check the invoice scores before keeping it).

In [ ]:
import os
from documind.ocr import ensemble, trocr
os.environ["DOCUMIND_OCR"] = "auto"
engine = trocr.get_engine()
print("engines:", "trained model on " + engine.device + " + Tesseract, together" if engine else "Tesseract only (no trained model found)")
sample = rows[:6]
if engine:
    neural = engine.read([Image.open(r["path"]).convert("RGB") for r in sample])
    from documind.ocr.baseline import read_tesseract
    for r, n in zip(sample, neural):
        t = read_tesseract(Image.open(r["path"]).convert("RGB"))
        pick = ensemble.choose({"trocr": n, "tesseract": t}, field=r["kind"])
        print(f"{r['kind']:16s} truth {r['text'][:30]!r:34s} trained {n[:30]!r:34s} tesseract {t[:30]!r:34s} -> {pick.source} ({pick.reason})")

### Notes
- The model is saved in `models/ocr_trocr` (about 250 MB, git-ignored like the other models), the training log in `reports/ocr_train_log.json`, the final comparison in `reports/ocr_report.json`.
- Command-line version: `python -m documind.data.prepare_ocr`, `python -m documind.training.train_ocr`, `python -m documind.training.evaluate_ocr`.
- **Licence:** `microsoft/trocr-*` models are MIT licensed.